# Using Lang Graph to generate questions about a database

**Goal**: to generate a set of questions and SQL pairs to ask about the Chinook database.

The questions and SQL queries should have dynamic fields, example json output:
``` json
{
    "question": "What is the total revenue generated by sales support agent <employee_last_name>?",
    "answer": "SELECT e.FirstName, e.LastName, SUM(i.Total) as total_revenue FROM Employee e JOIN Customer c ON e.EmployeeId = c.SupportRepId JOIN Invoice i ON c.CustomerId = i.CustomerId WHERE e.LastName = '<employee_last_name>';"
}
```

## Validation Loop

1. **Generate**: Create the initial batch of questions.
2. **Validate**: Test all SQL queries by attempting to execute them.
3. **Router**: This is a branching node:
    - If Error: Loop back and correct
    - If Clear: End
4. **Correct**: Pass the errors and broken pairs back to the LLM.

### Lang Chain **State**

Lang Chain handles state by passing an instance of a specific class between nodes and edges. For this test, I'm going to use the `GraphState` class.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, TypedDict

class QuestionSQLPair(BaseModel):
    question: str = Field(description="A synthetic user question about the data.")
    sql_query: str = Field(description="A valid SQL query to answer the question.")

# This is the one passed to the chain
class QuestionBatch(BaseModel):
    items: List[QuestionSQLPair] = Field(description="A list of question-query pairs")

# This represents the interface of the state
class GraphState(TypedDict):
    schema: str
    questions: List[QuestionSQLPair]
    erros: List[str]    # Errors will be stored here
    iterations: int     # This can be used to prevent infite loop

# Nodes

## Question Generation Node

This node uses `LangChain` to generate questions.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def generate_questions(state: GraphState):
    
    print("---GENERATING QUESTIONS---")
    schema = state["schema"]

    # It would be better to define this outside of the function body to avoid initializing it every time.
    # I'm placing it here to remind myself of how this whole thing works.
    prompt = ChatPromptTemplate.from_messages()

## SQL Valiation Node

This node is responsible for testing all the SQL queries, and returning the new `errors` piece of state.

In [3]:
from functions import execute_query, fill_sql_template


def validate_queries(state: GraphState):

    print("---VLIDATING QUERIES---")
    questions = state["questions"]
    errors = []

    for i, item in enumerate(questions):
        try:
            # Replace dynamic fields with real values. Ex: <artist> -> AC/DC
            converted_sql_query = fill_sql_template(item.sql_query)
            query_res = execute_query(converted_sql_query, "./data/Chinook_Sqlite.sqlite")
        except Exception as e:
            errors.append(f"Query {i+1} Failed: {str(e)}")

# Conditional Edges

Functionality can be added to edges

## Decicion Edge

This edge will be used to decide if the next step should be to go to the `correct` node, or the `END` node.

In [4]:
def decide_next_step(state: GraphState):
    if state["errors"] and state["iterations"] < 3:
        return "correct"    # Errors found and iterations are under 3, route to "correct" node
    return "end"

# The Graph

An instance of `StateGraph` is used to wire everything together.

In [ ]:
from langgraph.graph import StateGraph, END

# Pass in the state class during initialization
workflow = StateGraph(GraphState)

# Define the nodes
workflow.add_node("generate", generate_questions)
workflow.add_node("validate", validate_queries)
workflow.add_node("correct", correct_queries)

# Defin edges
# This is where the graph is stitched together and the flow is defined.
workflow.set_entry_point("generate")
workflow.add_edge("generate", "validate")   # After "generate" node runs, move to "validate" node.
workflow.add_conditional_edges("validate", decide_next_step, {"correct": "correct", "end": END})
workflow.add_edge("correct", "validate")    # Validate again after each correction

app = workflow.compile

## Visualizing the Graph

LangGraph has built in methods for rendering a visual representation of graphs.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback if dependencies are missing
    print(app.get_graph().draw_ascii())